In [3]:
import pandas as pd
import numpy as np
from datasets import load_dataset

D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = load_dataset("allenai/multi_lexsum", "v20230518", trust_remote_code=True)
dataset

Using the latest cached version of the module from C:\Users\wyh\.cache\huggingface\modules\datasets_modules\datasets\allenai--multi_lexsum\5424e588bc76c66297b1f63d985fcf095248d9a6959a2a36b70d9ccba8067131 (last modified on Thu May 28 21:13:26 2026) since it couldn't be found locally at allenai/multi_lexsum, or remotely on the Hugging Face Hub.


DatasetDict({
    train: Dataset({
        features: ['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short', 'summary/tiny', 'case_metadata'],
        num_rows: 3177
    })
    validation: Dataset({
        features: ['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short', 'summary/tiny', 'case_metadata'],
        num_rows: 454
    })
    test: Dataset({
        features: ['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short', 'summary/tiny', 'case_metadata'],
        num_rows: 908
    })
})

In [5]:
dataset.keys()

dict_keys(['train', 'validation', 'test'])

In [6]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

train_df.head()

,id,sources,sources_metadata,summary/long,summary/short,summary/tiny,case_metadata
0,EE-AL-0045,[Case 1:05-cv-00530-D Document 1-1 Filed 09/19...,"{'doc_id': ['EE-AL-0045-0001', 'EE-AL-0045-000...","On September 15, 2005, the Equal Employment Op...",Equal Employment Opportunity Commission brough...,NaN,{'case_name': 'EEOC v. House of Philadelphia C...
1,PB-NJ-0003,[Case 3:05-cv-01784-SRC-JJH Document 2 Filed 0...,"{'doc_id': ['PB-NJ-0003-0001', 'PB-NJ-0003-000...",NOTE: This is one of three identically named ...,The case was brought by a non-profit organizat...,NaN,{'case_name': 'Disability Rights New Jersey v....
2,EE-FL-0136,[Case 9:07-cv-80713-KAM Document 1 Entered on ...,"{'doc_id': ['EE-FL-0136-0001', 'EE-FL-0136-000...","On August 9, 2007, the United States Departmen...",NaN,NaN,{'case_name': 'United States v. Palm Beach Cou...
3,EE-CA-0305,[2006 WL 1787244\n2006 WL 1787244 (N.D.Cal.) (...,"{'doc_id': ['EE-CA-0305-0001', 'EE-CA-0305-000...","On May 11, 2006, African-American employees of...",This case was brought by African American empl...,NaN,{'case_name': 'Wynne v. McCormick & Schmick's ...
4,NH-NJ-0002,[IN THE UNITED STATES DISTRICT COURT FOR THE D...,"{'doc_id': ['NH-NJ-0002-0001', 'NH-NJ-0002-000...",Pursuant to the Civil Rights of Institutionali...,Pursuant to the Civil Rights of Institutionali...,NaN,"{'case_name': 'U.S. v. Mercer County, New Jers..."


In [7]:
train_df.columns

Index(['id', 'sources', 'sources_metadata', 'summary/long', 'summary/short',
       'summary/tiny', 'case_metadata'],
      dtype='str')

In [8]:
train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/validation.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

In [8]:
def build_case_text(row):
    return "\n\n".join(row["sources"])

def extract_case_metadata(row):
    meta = row["case_metadata"]
    return pd.Series({
        "case_name": meta.get("case_name"),
        "court": meta.get("court"),
        "date_filed": meta.get("date_filed"),
        "case_type": meta.get("case_type"),
        "class_action_sought": meta.get("class_action_sought"),
    })

train_clean = train_df.copy()

train_clean["full_text"] = train_clean.apply(build_case_text, axis=1)

metadata_df = train_clean.apply(extract_case_metadata, axis=1)
train_clean = pd.concat([train_clean, metadata_df], axis=1)

train_clean[[
    "id",
    "case_name",
    "case_type",
    "class_action_sought",
    "summary/long",
    "summary/short",
    "summary/tiny",
    "full_text"
]].head()

,id,case_name,case_type,class_action_sought,summary/long,summary/short,summary/tiny,full_text
0,EE-AL-0045,"EEOC v. House of Philadelphia Center, Inc.",Equal Employment,No,"On September 15, 2005, the Equal Employment Op...",Equal Employment Opportunity Commission brough...,NaN,Case 1:05-cv-00530-D Document 1-1 Filed 09/19/...
1,PB-NJ-0003,Disability Rights New Jersey v. Velez,Public Benefits / Government Services,No,NOTE: This is one of three identically named ...,The case was brought by a non-profit organizat...,NaN,Case 3:05-cv-01784-SRC-JJH Document 2 Filed 05...
2,EE-FL-0136,United States v. Palm Beach County,Equal Employment,No,"On August 9, 2007, the United States Departmen...",NaN,NaN,Case 9:07-cv-80713-KAM Document 1 Entered on F...
3,EE-CA-0305,Wynne v. McCormick & Schmick's Seafood Restara...,Equal Employment,Yes,"On May 11, 2006, African-American employees of...",This case was brought by African American empl...,NaN,2006 WL 1787244\n2006 WL 1787244 (N.D.Cal.) (T...
4,NH-NJ-0002,"U.S. v. Mercer County, New Jersey",Nursing Home Conditions,No,Pursuant to the Civil Rights of Institutionali...,Pursuant to the Civil Rights of Institutionali...,NaN,IN THE UNITED STATES DISTRICT COURT FOR THE DI...


In [13]:
train_clean["case_type"].value_counts(dropna=False)

case_type
Equal Employment                         1099
Immigration and/or the Border             367
Prison Conditions                         313
Jail Conditions                           223
Public Benefits / Government Services     204
Policing                                  143
Speech and Religious Freedom              134
Criminal Justice (Other)                  120
National Security                          94
Fair Housing/Lending/Insurance             84
Disability Rights-Pub. Accom.              71
Education                                  65
Juvenile Institution                       55
Election/Voting Rights                     51
Presidential/Gubernatorial Authority       31
Child Welfare                              31
Intellectual Disability (Facility)         29
Mental Health (Facility)                   23
Public Accomm./Contracting                 10
School Desegregation                        8
Public Housing                              8
Indigent Defense        

In [11]:
train_clean["class_action_sought"].value_counts(dropna=False)

class_action_sought
No         2109
Yes        1066
Unknown       2
Name: count, dtype: int64

In [12]:
train_clean["text_length"] = train_clean["full_text"].str.len()

train_clean["text_length"].describe()

count    3.177000e+03
mean     4.064372e+05
std      9.384189e+05
min      6.056000e+03
25%      5.947400e+04
50%      1.608180e+05
75%      4.039220e+05
max      2.350266e+07
Name: text_length, dtype: float64

In [13]:
train_clean.to_csv("../data/processed/train_clean.csv", index=False)

In [ ]:
## Export Processed Dataset

# The cleaned datasets are exported and used by downstream notebooks:

# - 02_baseline_summarization.ipynb
# - 03_classification_model.ipynb